# SGD Regressor: Efektivní lineární regrese pro velké datové sady

## Co je SGD Regressor?

SGD Regressor (Stochastic Gradient Descent Regressor) je lineární model pro regresi, který je trénován pomocí stochastického gradientního sestupu. Implementuje regularizovanou lineární regresi s možností použití různých ztrátových funkcí a penalizací. SGD Regressor je velmi efektivní, zejména pro velké datové sady, a nabízí flexibilitu volby ztrátové funkce.

### Klíčové vlastnosti:

1. **Efektivita** - Vhodný pro velké datové sady a online učení
2. **Flexibilita** - Podporuje různé ztrátové funkce a regularizace
3. **Rychlost** - Často mnohem rychlejší než standardní lineární regrese pro velké datasety
4. **Škálovatelnost** - Zpracovává data postupně, takže má nízké paměťové nároky

### Kdy použít SGD Regressor:

- Pro velké datové sady, kde klasické metody nejsou efektivní
- Při potřebě zpracovat data postupně (online learning)
- Pro rychlé prototypování různých modelů
- Když je důležitá výpočetní efektivita a rychlost

Pojďme si ukázat, jak tento algoritmus funguje v praxi.

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.datasets import fetch_california_housing, load_diabetes
import time

# Pro grafy
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Pro reprodukovatelnost
np.random.seed(42)

## 1. Stochastický gradientní sestup (SGD) - princip

Stochastický gradientní sestup je optimalizační algoritmus, který aktualizuje parametry modelu postupně pro každý vzorek nebo malou dávku vzorků (na rozdíl od běžného gradientního sestupu, který používá všechna data najednou).

### Základní kroky SGD:

1. Inicializace vah náhodně nebo na nulu
2. Pro každou epochu (průchod daty):
   - Zamíchání dat
   - Pro každý vzorek nebo mini-dávku:
     - Výpočet predikce
     - Výpočet chyby
     - Aktualizace vah podle gradientu chyby a learning rate
3. Opakování, dokud není dosaženo konvergence nebo maximálního počtu epoch

Výhodou je, že algoritmus nemusí načítat všechna data do paměti najednou a je výpočetně efektivní.

## 2. Základní implementace SGD Regressoru

Nejprve načteme dataset California Housing, který obsahuje data o cenách domů v Kalifornii. Je to středně velký dataset, takže uvidíme výhody SGD oproti standardní lineární regresi.

In [ ]:
# Načtení datasetu
housing = fetch_california_housing()
X, y = housing.data, housing.target

# Základní informace o datasetu
print(f"Tvar datasetu: {X.shape}")
print(f"Příznaky: {housing.feature_names}")
print(f"Cíl: Medián hodnoty domů v $100,000")

# Rozdělení na trénovací a testovací data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardizace dat (důležité pro SGD)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Implementace základního SGD Regressoru
sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)

# Měření času trénování
start_time = time.time()
sgd_reg.fit(X_train_scaled, y_train)
sgd_training_time = time.time() - start_time

# Predikce na testovacích datech
y_pred_sgd = sgd_reg.predict(X_test_scaled)

# Vyhodnocení výkonu
mse_sgd = mean_squared_error(y_test, y_pred_sgd)
r2_sgd = r2_score(y_test, y_pred_sgd)

print(f"SGD Regressor výsledky:")
print(f"MSE: {mse_sgd:.4f}")
print(f"RMSE: {np.sqrt(mse_sgd):.4f}")
print(f"R^2: {r2_sgd:.4f}")
print(f"Čas trénování: {sgd_training_time:.4f} sekund")

### Porovnání se standardní lineární regresí

Pro srovnání implementujeme standardní lineární regresi na stejných datech:

In [ ]:
# Implementace standardní lineární regrese
lr = LinearRegression()

# Měření času trénování
start_time = time.time()
lr.fit(X_train_scaled, y_train)
lr_training_time = time.time() - start_time

# Predikce
y_pred_lr = lr.predict(X_test_scaled)

# Vyhodnocení výkonu
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Standardní lineární regrese výsledky:")
print(f"MSE: {mse_lr:.4f}")
print(f"RMSE: {np.sqrt(mse_lr):.4f}")
print(f"R^2: {r2_lr:.4f}")
print(f"Čas trénování: {lr_training_time:.4f} sekund")

# Porovnání časů trénování
print(f"\nPorovnání časů trénování:")
print(f"SGD Regressor: {sgd_training_time:.4f} s")
print(f"Lineární regrese: {lr_training_time:.4f} s")
print(f"Poměr (LR/SGD): {lr_training_time/sgd_training_time:.2f}x")

In [ ]:
# Vizualizace skutečných vs. predikovaných hodnot
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_sgd, alpha=0.5)
plt.plot([0, 5], [0, 5], 'r--')
plt.xlabel('Skutečné hodnoty')
plt.ylabel('Predikované hodnoty')
plt.title('SGD Regressor: Skutečné vs. Predikované hodnoty')

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_lr, alpha=0.5)
plt.plot([0, 5], [0, 5], 'r--')
plt.xlabel('Skutečné hodnoty')
plt.ylabel('Predikované hodnoty')
plt.title('Standardní lineární regrese: Skutečné vs. Predikované hodnoty')

plt.tight_layout()
plt.show()

### Analýza reziduí

Rezidua jsou rozdíly mezi skutečnými a predikovanými hodnotami. Pro dobrý model by rezidua měla být náhodně distribuována kolem nuly.

In [ ]:
# Výpočet reziduí
residuals_sgd = y_test - y_pred_sgd
residuals_lr = y_test - y_pred_lr

# Vizualizace reziduí
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(y_pred_sgd, residuals_sgd, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predikované hodnoty')
plt.ylabel('Rezidua')
plt.title('SGD Regressor: Rezidua')

plt.subplot(1, 2, 2)
plt.scatter(y_pred_lr, residuals_lr, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predikované hodnoty')
plt.ylabel('Rezidua')
plt.title('Standardní lineární regrese: Rezidua')

plt.tight_layout()
plt.show()

# Distribuce reziduí
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(residuals_sgd, kde=True)
plt.xlabel('Rezidua')
plt.title('SGD Regressor: Distribuce reziduí')

plt.subplot(1, 2, 2)
sns.histplot(residuals_lr, kde=True)
plt.xlabel('Rezidua')
plt.title('Standardní lineární regrese: Distribuce reziduí')

plt.tight_layout()
plt.show()

## 3. Regularizace v SGD Regressoru

SGD Regressor umožňuje použití různých typů regularizace, které pomáhají předcházet přeučení:

1. **L1 regularizace (Lasso)** - Penalizuje absolutní hodnotu vah, vede k řídkým modelům
2. **L2 regularizace (Ridge)** - Penalizuje čtverce vah, vede k menším vahám
3. **ElasticNet** - Kombinace L1 a L2 regularizace

Podívejme se, jak lze nastavit regularizaci a jak ovlivňuje výkon modelu:

In [ ]:
# SGD s různými typy regularizace
regularization_types = {
    'Bez regularizace': SGDRegressor(penalty=None, max_iter=1000, random_state=42),
    'L1 (Lasso)': SGDRegressor(penalty='l1', alpha=0.01, max_iter=1000, random_state=42),
    'L2 (Ridge)': SGDRegressor(penalty='l2', alpha=0.01, max_iter=1000, random_state=42),
    'ElasticNet': SGDRegressor(penalty='elasticnet', alpha=0.01, l1_ratio=0.5, max_iter=1000, random_state=42)
}

# Vyhodnocení modelů
results = {}
coefs_df = pd.DataFrame()

for name, model in regularization_types.items():
    # Trénování
    model.fit(X_train_scaled, y_train)
    
    # Predikce
    y_pred = model.predict(X_test_scaled)
    
    # Metriky
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R^2': r2
    }
    
    # Uložení koeficientů
    coefs_df[name] = model.coef_

# Výpis výsledků
results_df = pd.DataFrame(results).T
print("Výsledky různých typů regularizace:")
print(results_df)

In [ ]:
# Vizualizace koeficientů pro různé typy regularizace
coefs_df.index = housing.feature_names
ax = coefs_df.plot(kind='bar', figsize=(14, 8))
plt.title('Porovnání koeficientů pro různé typy regularizace')
plt.xlabel('Příznaky')
plt.ylabel('Koeficienty')
plt.grid(True, axis='y')
plt.legend(title='Typ regularizace')
plt.tight_layout()
plt.show()

### Vliv parametru alpha (síla regularizace)

Alpha je parametr, který řídí sílu regularizace. Podívejme se, jak ovlivňuje výkon modelu:

In [ ]:
# Zkoumání vlivu parametru alpha na výkon modelu
alphas = [0.0001, 0.001, 0.01, 0.1, 1.0]
alpha_results = {}

for penalty in ['l1', 'l2', 'elasticnet']:
    alpha_results[penalty] = {'train_scores': [], 'test_scores': [], 'alphas': alphas}
    
    for alpha in alphas:
        # Nastavení modelu
        if penalty == 'elasticnet':
            sgd = SGDRegressor(penalty=penalty, alpha=alpha, l1_ratio=0.5, max_iter=1000, random_state=42)
        else:
            sgd = SGDRegressor(penalty=penalty, alpha=alpha, max_iter=1000, random_state=42)
        
        # Trénování
        sgd.fit(X_train_scaled, y_train)
        
        # Vyhodnocení na trénovacích a testovacích datech
        train_score = r2_score(y_train, sgd.predict(X_train_scaled))
        test_score = r2_score(y_test, sgd.predict(X_test_scaled))
        
        alpha_results[penalty]['train_scores'].append(train_score)
        alpha_results[penalty]['test_scores'].append(test_score)

# Vizualizace vlivu alpha
plt.figure(figsize=(15, 10))

for i, (penalty, results) in enumerate(alpha_results.items()):
    plt.subplot(2, 2, i+1)
    plt.semilogx(results['alphas'], results['train_scores'], 'b-o', label='Trénink R²')
    plt.semilogx(results['alphas'], results['test_scores'], 'r-o', label='Test R²')
    plt.xlabel('Alpha (síla regularizace)')
    plt.ylabel('R² skóre')
    plt.title(f'Vliv alpha na výkon modelu s {penalty} regularizací')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

## 4. Různé ztrátové funkce

SGD Regressor může používat různé ztrátové funkce, které definují, jak model měří chyby predikce během trénování. Scikit-learn nabízí následující možnosti:

- `'squared_error'` (výchozí) - Mean Squared Error (MSE)
- `'huber'` - Huberova ztráta, která je méně citlivá na odlehlé hodnoty než MSE
- `'epsilon_insensitive'` - Ignoruje chyby menší než epsilon (podobné jako v SVR)
- `'squared_epsilon_insensitive'` - Druhá mocnina předchozí funkce

Pojďme porovnat jejich výkon:

In [ ]:
# Porovnání různých ztrátových funkcí
loss_functions = {
    'squared_error': SGDRegressor(loss='squared_error', penalty='l2', alpha=0.01, max_iter=1000, random_state=42),
    'huber': SGDRegressor(loss='huber', penalty='l2', alpha=0.01, max_iter=1000, random_state=42, epsilon=0.1),
    'epsilon_insensitive': SGDRegressor(loss='epsilon_insensitive', penalty='l2', alpha=0.01, max_iter=1000, random_state=42, epsilon=0.1),
    'squared_epsilon_insensitive': SGDRegressor(loss='squared_epsilon_insensitive', penalty='l2', alpha=0.01, max_iter=1000, random_state=42, epsilon=0.1)
}

loss_results = {}

for name, model in loss_functions.items():
    # Trénování
    model.fit(X_train_scaled, y_train)
    
    # Predikce
    y_pred = model.predict(X_test_scaled)
    
    # Metriky
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    loss_results[name] = {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mae,
        'R^2': r2
    }

# Výpis výsledků
loss_results_df = pd.DataFrame(loss_results).T
print("Výsledky různých ztrátových funkcí:")
print(loss_results_df)

In [ ]:
# Vizualizace výsledků pro různé ztrátové funkce
metrics = ['MSE', 'MAE', 'R^2']
plt.figure(figsize=(15, 10))

for i, metric in enumerate(metrics):
    plt.subplot(2, 2, i+1)
    plt.bar(loss_results_df.index, loss_results_df[metric])
    plt.title(f'Porovnání {metric} pro různé ztrátové funkce')
    plt.ylabel(metric)
    plt.xticks(rotation=45)
    plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 5. Ukázka na datech s odlehlými hodnotami

Některé ztrátové funkce, jako je Huberova ztráta, jsou navrženy tak, aby byly odolnější vůči odlehlým hodnotám. Podívejme se, jak si různé ztrátové funkce vedou na datech s odlehlými hodnotami.

In [ ]:
# Vytvoření dat s odlehlými hodnotami
np.random.seed(42)
X = np.random.rand(200, 1) * 10 - 5  # Data v rozsahu [-5, 5]
y = 0.5 * X.ravel() + 1 + 0.5 * np.random.randn(200)  # Lineární vztah s šumem

# Přidání několika odlehlých hodnot
outlier_indices = np.random.choice(200, 20, replace=False)
y[outlier_indices] = y[outlier_indices] + 5 * np.random.randn(20)

# Rozdělení na trénovací a testovací data
X_train_outliers, X_test_outliers, y_train_outliers, y_test_outliers = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Vizualizace dat
plt.figure(figsize=(10, 6))
plt.scatter(X_train_outliers, y_train_outliers, alpha=0.6, label='Trénovací data')
plt.scatter(X_test_outliers, y_test_outliers, alpha=0.6, label='Testovací data', color='red')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Data s odlehlými hodnotami')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Porovnání ztrátových funkcí na datech s odlehlými hodnotami
outlier_models = {
    'squared_error': SGDRegressor(loss='squared_error', max_iter=1000, random_state=42),
    'huber': SGDRegressor(loss='huber', epsilon=0.1, max_iter=1000, random_state=42),
    'epsilon_insensitive': SGDRegressor(loss='epsilon_insensitive', epsilon=0.1, max_iter=1000, random_state=42)
}

# Škálování dat není nutné pro tento jednoduchý příklad
plt.figure(figsize=(14, 8))

# Trénování modelů a vizualizace výsledků
x_plot = np.linspace(-5, 5, 100).reshape(-1, 1)

for i, (name, model) in enumerate(outlier_models.items()):
    model.fit(X_train_outliers, y_train_outliers)
    y_pred = model.predict(X_test_outliers)
    mse = mean_squared_error(y_test_outliers, y_pred)
    y_plot = model.predict(x_plot)
    
    plt.subplot(1, 3, i+1)
    plt.scatter(X_train_outliers, y_train_outliers, alpha=0.4, label='Trénovací data')
    plt.scatter(X_test_outliers, y_test_outliers, alpha=0.4, label='Testovací data', color='red')
    plt.plot(x_plot, y_plot, 'g-', linewidth=2, label=f'Model (MSE: {mse:.4f})')
    plt.title(f'SGD s {name} ztrátou')
    plt.xlabel('X')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

## 6. Optimální parametry pomocí grid search

Pro nalezení optimálních hyperparametrů SGD Regressoru můžeme použít techniku grid search:

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definice pipeline se standardizací a SGD Regressorem
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDRegressor(random_state=42))
])

# Parametry pro grid search
param_grid = {
    'sgd__loss': ['squared_error', 'huber'],
    'sgd__penalty': [None, 'l2', 'l1', 'elasticnet'],
    'sgd__alpha': [0.0001, 0.001, 0.01],
    'sgd__epsilon': [0.05, 0.1, 0.2],  # Pro Huberovu ztrátu
    'sgd__max_iter': [1000]
}

# Nastavení a spuštění grid search
grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1
)

print("Spouštím grid search (může trvat několik minut)...")
grid_search.fit(X_train, y_train)

# Výsledky
print(f"\nNejlepší parametry: {grid_search.best_params_}")
print(f"Nejlepší skóre (neg. MSE): {grid_search.best_score_}")
print(f"Nejlepší model: {grid_search.best_estimator_}")

# Vyhodnocení nejlepšího modelu
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mse_best = mean_squared_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

print(f"\nVýkon nejlepšího modelu na testovacích datech:")
print(f"MSE: {mse_best:.4f}")
print(f"RMSE: {np.sqrt(mse_best):.4f}")
print(f"R^2: {r2_best:.4f}")

## 7. Online učení s SGD Regressorem

Jednou z klíčových výhod SGD Regressoru je jeho schopnost učit se inkrementálně, což je užitečné pro velké datasety nebo streamování dat. Podívejme se, jak lze SGD Regressor použít pro online učení:

In [ ]:
# Demonstrace online učení
from sklearn.base import clone

# Načtení datasetu diabetes pro změnu
diabetes = load_diabetes()
X_diabetes, y_diabetes = diabetes.data, diabetes.target

# Standardizace dat
scaler = StandardScaler()
X_diabetes_scaled = scaler.fit_transform(X_diabetes)

# Inicializace modelů
sgd_batch = SGDRegressor(max_iter=1, learning_rate='constant', eta0=0.01, random_state=42)
sgd_online = clone(sgd_batch)

# Simulace dávkového učení (jednorázové trénování na všech datech) vs. online učení
n_samples = X_diabetes_scaled.shape[0]
batch_size = 50
n_batches = n_samples // batch_size

# Online learning errors
online_errors = []
batch_errors = []

# Rozdělení dat do dávek
for epoch in range(10):  # 10 průchodů daty pro online učení
    # Zamíchání dat pro každou epochu
    indices = np.random.permutation(n_samples)
    X_shuffled = X_diabetes_scaled[indices]
    y_shuffled = y_diabetes[indices]
    
    # Online learning po malých dávkách
    for i in range(0, n_samples, batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]
        
        # Partial_fit pro inkrementální učení
        sgd_online.partial_fit(X_batch, y_batch)
        
        # Počítání MSE po každé dávce
        y_pred_online = sgd_online.predict(X_diabetes_scaled)
        online_errors.append(mean_squared_error(y_diabetes, y_pred_online))
    
    # Trénování batch modelu po každé epoše pro srovnání
    if epoch == 0:
        sgd_batch.fit(X_diabetes_scaled, y_diabetes)  # Jednorázové trénování
    
    y_pred_batch = sgd_batch.predict(X_diabetes_scaled)
    batch_errors.append(mean_squared_error(y_diabetes, y_pred_batch))

# Vizualizace křivky učení
plt.figure(figsize=(12, 6))
plt.plot(online_errors, label='Online učení (po dávkách)')
plt.axhline(y=batch_errors[0], color='r', linestyle='--', 
            label='Dávkové učení (jednorázové)')
plt.xlabel('Počet dávek')
plt.ylabel('MSE')
plt.title('Křivka učení: Online vs. Batch')
plt.legend()
plt.grid(True)
plt.show()

## 8. Warm start a pokračování v trénování

SGD Regressor umožňuje pokračovat v trénování s novými daty pomocí parametru `warm_start`. To je užitečné, když máte nová data nebo chcete pokračovat v trénování po více epoch:

In [ ]:
# Ukázka warm start
sgd_warm = SGDRegressor(max_iter=5, warm_start=True, random_state=42)

# Trénování postupně po více epoch
n_epochs = 20
training_errors = []

for epoch in range(n_epochs):
    sgd_warm.fit(X_train_scaled, y_train)  # Díky warm_start pokračuje v trénování
    y_pred = sgd_warm.predict(X_train_scaled)
    mse = mean_squared_error(y_train, y_pred)
    training_errors.append(mse)

# Vizualizace
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_epochs + 1), training_errors, 'b-o')
plt.xlabel('Epocha')
plt.ylabel('MSE na trénovacích datech')
plt.title('Křivka učení s warm start')
plt.grid(True)
plt.show()

## 9. Porovnání SGD Regressoru s jinými regresními modely

Srovnejme výkon a rychlost SGD Regressoru s jinými regresními modely v scikit-learn:

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.ensemble import RandomForestRegressor

# Definice modelů
models = {
    'SGD Regressor': SGDRegressor(max_iter=1000, random_state=42),
    'Standardní lineární regrese': LinearRegression(),
    'Ridge regrese': Ridge(random_state=42),
    'Lasso regrese': Lasso(random_state=42),
    'ElasticNet': ElasticNet(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Vyhodnocení modelů
model_results = []

for name, model in models.items():
    # Měření času trénování
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    training_time = time.time() - start_time
    
    # Měření času predikce
    start_time = time.time()
    y_pred = model.predict(X_test_scaled)
    prediction_time = time.time() - start_time
    
    # Metriky
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    model_results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R^2': r2,
        'Training Time': training_time,
        'Prediction Time': prediction_time
    })

# Výpis výsledků
model_results_df = pd.DataFrame(model_results)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("Porovnání regresních modelů:")
print(model_results_df)

In [ ]:
# Vizualizace výsledků
plt.figure(figsize=(15, 12))

# RMSE
plt.subplot(2, 2, 1)
sns.barplot(x='Model', y='RMSE', data=model_results_df)
plt.xticks(rotation=45, ha='right')
plt.title('RMSE pro různé regresní modely')
plt.grid(True, axis='y')

# R^2
plt.subplot(2, 2, 2)
sns.barplot(x='Model', y='R^2', data=model_results_df)
plt.xticks(rotation=45, ha='right')
plt.title('R² pro různé regresní modely')
plt.grid(True, axis='y')

# Čas trénování
plt.subplot(2, 2, 3)
sns.barplot(x='Model', y='Training Time', data=model_results_df)
plt.xticks(rotation=45, ha='right')
plt.title('Čas trénování (s)')
plt.grid(True, axis='y')

# Čas predikce
plt.subplot(2, 2, 4)
sns.barplot(x='Model', y='Prediction Time', data=model_results_df)
plt.xticks(rotation=45, ha='right')
plt.title('Čas predikce (s)')
plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 10. Shrnutí: Kdy použít SGD Regressor

### Výhody:

1. **Efektivita** - Velmi efektivní pro velké datasety díky postupnému zpracování dat
2. **Flexibilita** - Různé ztrátové funkce a regularizace
3. **Online učení** - Možnost postupného učení s novými daty
4. **Nízká paměťová náročnost** - Nepotřebuje najednou celý dataset v paměti
5. **Regularizace** - Snadná implementace regularizace pro prevenci přeučení

### Nevýhody:

1. **Citlivost na škálování** - Vyžaduje standardizaci dat
2. **Ladění hyperparametrů** - Více parametrů k ladění (learning rate, regularizace)
3. **Konvergence** - Může vyžadovat více iterací pro konvergenci
4. **Stochastická povaha** - Výsledky se mohou mírně lišit mezi běhy

### Doporučení pro použití:

- Pro velké datasety, kde jsou standardní metody pomalé
- Pro online learning nebo streamování dat
- Když máte omezené výpočetní prostředky (paměť)
- Pro rychlé prototypování lineárních modelů
- Při potřebě flexibility ve ztrátových funkcích a regularizacích

### Praktické tipy:

- Vždy standardizujte data před použitím SGD Regressoru
- Experimentujte s různými ztrátovými funkcemi, zejména Huber pro data s odlehlými hodnotami
- Nastavte `max_iter` dostatečně vysoko pro konvergenci
- Použijte `warm_start=True`, pokud chcete pokračovat v trénování se stejným modelem
- Regularizace je důležitá - experimentujte s `alpha` pro prevenci přeučení
- Pro inkrementální učení používejte `partial_fit` místo `fit`

SGD Regressor je výkonný nástroj pro regresní analýzu, který kombinuje efektivitu s flexibilitou, což ho činí ideální volbou pro mnoho praktických aplikací strojového učení.